In [ ]:
pip install xgboost

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import KFold, cross_val_score, RandomizedSearchCV
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import OrdinalEncoder
from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor

import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Ganti path sesuai dengan lokasi file Anda
train_df = pd.read_csv('../data/train.csv')
test_df = pd.read_csv('../data/test.csv')

# 1. FILTER OUTLIER DI AWAL (Sebelum memisahkan apapun)
train_df = train_df[train_df.GrLivArea < 4500].reset_index(drop=True)

# Pisahkan ID dan Target (SalePrice)
train_ID = train_df['Id']
test_ID = test_df['Id']

# 2. LOG TRANSFORMATION PADA TARGET
y_train = np.log1p(train_df['SalePrice'])

# 3. GABUNGKAN DATA TANPA TARGET LEAKAGE
ntrain = train_df.shape[0]
ntest = test_df.shape[0]

# Triknya: Kita drop Id dan SalePrice HANYA SAAT MENGGABUNGKAN ke all_data
# Dengan begini, train_df tetap punya SalePrice untuk grafik EDA di bawahnya
all_data = pd.concat((
    train_df.drop(['Id', 'SalePrice'], axis=1), 
    test_df.drop('Id', axis=1)
)).reset_index(drop=True)

print(f"Total ukuran data gabungan: {all_data.shape}")

### Exploratory data analysis

In [ ]:
sns.set_theme(style="whitegrid")

In [ ]:
# EDA 1: Target Variable Distribution
plt.figure(figsize=(8, 5))
sns.histplot(train_df['SalePrice'], kde=True, bins=40, color='blue')
plt.title('Distribution of SalePrice')
plt.xlabel('SalePrice')
plt.ylabel('Frequency')
plt.tight_layout()
plt.show()

In [ ]:
# EDA 2: Checking GrLivArea Outliers & Relationship with SalePrice
plt.figure(figsize=(8, 5))
sns.scatterplot(x=train_df['GrLivArea'], y=train_df['SalePrice'], alpha=0.6, color='green')
plt.title('SalePrice vs GrLivArea (Above Ground Living Area)')
plt.xlabel('GrLivArea (sq ft)')
plt.ylabel('SalePrice')
plt.tight_layout()
plt.show()

In [ ]:
# EDA 3: Correlation Heatmap
# Find variables with highest absolute correlation (>0.5) to the target
numerical_cols = train_df.select_dtypes(include=[np.number])
corr_matrix = numerical_cols.corr()
top_corr_features = corr_matrix.index[abs(corr_matrix["SalePrice"]) > 0.5]

plt.figure(figsize=(10, 8))
sns.heatmap(train_df[top_corr_features].corr(), annot=True, cmap='coolwarm', fmt=".2f", linewidths=.5)
plt.title('Correlation Heatmap (Highly Correlated Features)')
plt.tight_layout()
plt.show()

In [ ]:
# EDA 4: Analisis Missing Values (Nilai Kosong)
# Menghitung persentase missing values tiap kolom
missing_counts = train_df.isnull().sum()
missing_percent = (missing_counts / len(train_df)) * 100
missing_data = pd.DataFrame({'Missing_Ratio': missing_percent})
missing_data = missing_data[missing_data['Missing_Ratio'] > 0].sort_values(by='Missing_Ratio', ascending=False)

# Visualisasi Barplot
plt.figure(figsize=(10, 6))
sns.barplot(x=missing_data.index, y=missing_data['Missing_Ratio'], palette='viridis')
plt.xticks(rotation=90)
plt.xlabel('Features (Fitur)', fontsize=12)
plt.ylabel('Persentase Data Kosong (%)', fontsize=12)
plt.title('Fitur dengan Persentase Data Kosong Tertinggi', fontsize=15)
plt.tight_layout()
plt.show()

In [ ]:
# EDA 5: Analisis Kemencengan (Skewness) pada SalePrice
# Membandingkan distribusi SalePrice asli dan setelah di-Log
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: SalePrice Asli
sns.histplot(train_df['SalePrice'], kde=True, ax=axes[0], color='blue', bins=40)
axes[0].set_title('Distribusi Asli SalePrice (Right-Skewed)')

# Plot 2: SalePrice setelah Log Transformation (np.log1p untuk log(1+x))
sns.histplot(np.log1p(train_df['SalePrice']), kde=True, ax=axes[1], color='purple', bins=40)
axes[1].set_title('Distribusi Log(SalePrice) (Lebih Normal)')

plt.tight_layout()
plt.show()

print("Catatan: Jika metrik evaluasi model kurang bagus, kita bisa mempertimbangkan untuk mentransformasi target menggunakan Log.")

In [ ]:
# EDA 6: Kualitas Keseluruhan (OverallQual) vs Harga (SalePrice)
plt.figure(figsize=(10, 6))
# Menggunakan boxplot untuk melihat sebaran harga di setiap tingkat kualitas
sns.boxplot(x='OverallQual', y='SalePrice', data=train_df, palette='coolwarm')

plt.title('Hubungan Kualitas Rumah (OverallQual) dengan Harga (SalePrice)', fontsize=15)
plt.xlabel('Tingkat Kualitas (1 = Terburuk, 10 = Terbaik)', fontsize=12)
plt.ylabel('Harga Rumah (SalePrice)', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# EDA 7: Tahun Dibangun (YearBuilt) vs Harga (SalePrice)
# Melihat tren apakah rumah yang lebih baru cenderung memiliki harga yang lebih tinggi dibandingkan rumah tua.
plt.figure(figsize=(12, 6))
# Line plot / Scatter plot untuk melihat tren tahun
sns.scatterplot(x='YearBuilt', y='SalePrice', data=train_df, alpha=0.5, color='orange')

plt.title('Tren Harga Rumah Berdasarkan Tahun Dibangun', fontsize=15)
plt.xlabel('Tahun Dibangun (YearBuilt)', fontsize=12)
plt.ylabel('Harga Rumah (SalePrice)', fontsize=12)
plt.tight_layout()
plt.show()

### HANDLING MISSING DATA

In [ ]:
# Group A: NA berarti "None" (Fasilitas tidak ada)
cols_none = ['PoolQC', 'MiscFeature', 'Alley', 'Fence', 'FireplaceQu', 
             'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond', 
             'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2', 
             'MasVnrType']

for col in cols_none:
    all_data[col] = all_data[col].fillna('None')

# Group B: NA berarti 0 (Area atau ukuran 0 karena tidak ada fasilitasnya)
cols_zero = ['GarageYrBlt', 'GarageArea', 'GarageCars', 
             'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF', 
             'BsmtFullBath', 'BsmtHalfBath', 'MasVnrArea']

for col in cols_zero:
    all_data[col] = all_data[col].fillna(0)

# Group C: Sisanya (Benar-benar data yang hilang / missing values sesungguhnya)
# Isi dengan modus (nilai yang paling sering muncul)
cols_mode = ['MSZoning', 'Electrical', 'KitchenQual', 'Exterior1st', 
             'Exterior2nd', 'SaleType', 'Utilities']
for col in cols_mode:
    all_data[col] = all_data[col].fillna(all_data[col].mode()[0])

# LotFrontage biasanya mirip dengan rumah di lingkungan yang sama, isi dengan median lingkungan sekitarnya
all_data["LotFrontage"] = all_data.groupby("Neighborhood")["LotFrontage"].transform(
    lambda x: x.fillna(x.median()))

# Functional: Data description bilang NA berarti Typ (Typical)
all_data["Functional"] = all_data["Functional"].fillna("Typ")

# Cek apakah masih ada missing value
print("Total missing values tersisa:", all_data.isnull().sum().sum())

### FEATURE ENGINEERING ###

In [ ]:
# ==========================================
# SEL 3.5: FEATURE ENGINEERING
# ==========================================

# 1. TOTAL LUAS BANGUNAN (Sangat Krusial)
# Pembeli rumah biasanya melihat total luas keseluruhan, bukan terpisah per lantai
all_data['TotalSF'] = all_data['TotalBsmtSF'] + all_data['1stFlrSF'] + all_data['2ndFlrSF']

# 2. TOTAL KAMAR MANDI
# Menggabungkan semua kamar mandi (kamar mandi setengah dihitung 0.5)
all_data['Total_Bathrooms'] = (all_data['FullBath'] + (0.5 * all_data['HalfBath']) + 
                               all_data['BsmtFullBath'] + (0.5 * all_data['BsmtHalfBath']))

# 3. TOTAL LUAS TERAS (Porch)
# Menggabungkan semua jenis teras/dek kayu menjadi satu metrik luas area luar ruangan
all_data['Total_PorchSF'] = (all_data['OpenPorchSF'] + all_data['3SsnPorch'] + 
                             all_data['EnclosedPorch'] + all_data['ScreenPorch'] + 
                             all_data['WoodDeckSF'])

# 4. UMUR RUMAH (Kalkulasi saat rumah dijual)
# Mengubah YrSold ke angka sementara untuk dihitung
all_data['YrSold_num'] = all_data['YrSold'].astype(int)

# Menghitung umur rumah dan umur sejak renovasi terakhir
all_data['HouseAge'] = all_data['YrSold_num'] - all_data['YearBuilt']
all_data['HouseRemodelAge'] = all_data['YrSold_num'] - all_data['YearRemodAdd']

# Hapus kolom YrSold_num karena sudah tidak dipakai
all_data.drop('YrSold_num', axis=1, inplace=True)

# 5. FITUR BINER (Indikator Ya/Tidak)
# Model pohon (seperti XGBoost/Random Forest) sangat menyukai fitur biner (1=Ya, 0=Tidak)
all_data['HasPool'] = all_data['PoolArea'].apply(lambda x: 1 if x > 0 else 0)
all_data['Has2ndFloor'] = all_data['2ndFlrSF'].apply(lambda x: 1 if x > 0 else 0)
all_data['HasGarage'] = all_data['GarageArea'].apply(lambda x: 1 if x > 0 else 0)
all_data['HasBsmt'] = all_data['TotalBsmtSF'].apply(lambda x: 1 if x > 0 else 0)
all_data['HasFireplace'] = all_data['Fireplaces'].apply(lambda x: 1 if x > 0 else 0)

print(f"Bentuk data setelah Feature Engineering: {all_data.shape}")

In [ ]:
# Mengubah fitur numerik yang sebenarnya adalah kategori menjadi string dulu
all_data['MSSubClass'] = all_data['MSSubClass'].apply(str)
all_data['OverallCond'] = all_data['OverallCond'].apply(str)
all_data['YrSold'] = all_data['YrSold'].astype(str)
all_data['MoSold'] = all_data['MoSold'].astype(str)

# 1. ORDINAL ENCODING (Kategori yang ada tingkatannya)
# Mapping nilai kualitas menjadi angka
qual_map = {'None': 0, 'Po': 1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5}
ordinal_cols = ['BsmtQual', 'BsmtCond', 'FireplaceQu', 'GarageQual', 'GarageCond', 
                'ExterQual', 'ExterCond', 'HeatingQC', 'KitchenQual', 'PoolQC']

for col in ordinal_cols:
    all_data[col] = all_data[col].map(qual_map)

# 2. ONE-HOT ENCODING (Kategori tanpa tingkatan)
all_data = pd.get_dummies(all_data)

print(f"Bentuk data setelah Encoding: {all_data.shape}")

In [ ]:

mvp_features = [
    'TotalSF', 'LotArea', 'YearBuilt', 'BedroomAbvGr',
    'Total_Bathrooms', 'GarageCars'
]

all_data_mvp = all_data[mvp_features]

X_train = all_data_mvp[:ntrain]
X_test = all_data_mvp[ntrain:]
# Siapkan K-Fold Cross Validation
kf = KFold(n_splits=5, shuffle=True, random_state=42)


# Fungsi untuk menghitung RMSE (Root Mean Squared Error)
def cv_rmse(model):
    # Menggunakan neg_mean_squared_error, lalu di-akar kuadratkan
    rmse = np.sqrt(-cross_val_score(model, X_train, y_train, scoring="neg_mean_squared_error", cv=kf))
    return rmse

In [ ]:
# XGBoost
model_xgb = XGBRegressor(random_state=42)
score_xgb = cv_rmse(model_xgb)
print(f"XGBoost Baseline RMSE: {score_xgb.mean():.4f} (± {score_xgb.std():.4f})")

# Random Forest
model_rf = RandomForestRegressor(random_state=42)
score_rf = cv_rmse(model_rf)
print(f"Random Forest Baseline RMSE: {score_rf.mean():.4f} (± {score_rf.std():.4f})")

### XGBOOST HYPERPARAMETER TUNING ###

In [ ]:
param_grid = {
    'n_estimators': [500, 1000, 1500],
    'learning_rate': [0.01, 0.05, 0.1],
    'max_depth': [3, 4, 5],
    'subsample': [0.7, 0.8, 0.9],
    'colsample_bytree': [0.7, 0.8, 0.9]
}

random_search = RandomizedSearchCV(
    estimator=XGBRegressor(random_state=42),
    param_distributions=param_grid,
    n_iter=20,
    cv=kf,
    scoring='neg_mean_squared_error',
    random_state=42,
    n_jobs=-1
)

random_search.fit(X_train, y_train)

print('Parameter Terbaik XGBoost:', random_search.best_params_)
print('RMSE Terbaik XGBoost:', np.sqrt(-random_search.best_score_))

best_xgb_model = random_search.best_estimator_

In [ ]:
import joblib

best_xgb_model.fit(X_train, y_train)

xgb_log_predictions = best_xgb_model.predict(X_test)
xgb_final_predictions = np.expm1(xgb_log_predictions)

xgb_submission = pd.DataFrame({'Id': test_ID, 'SalePrice': xgb_final_predictions})
xgb_submission.to_csv('../outputs/submission_xgboost.csv', index=False)

joblib.dump(best_xgb_model, '../models/model_xgboost_rumah.pkl')
print('XGBoost model saved as models/model_xgboost_rumah.pkl')
print('Submission saved as submission_xgboost.csv')

### RANDOM FOREST HYPERPARAMETER TUNING ###

In [ ]:
rf_param_grid = {
    'n_estimators': [500, 1000, 1500],
    'max_depth': [3, 5, 10, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2']
}

rf_random_search = RandomizedSearchCV(
    estimator=RandomForestRegressor(random_state=42),
    param_distributions=rf_param_grid,
    n_iter=20,
    cv=kf,
    scoring='neg_mean_squared_error',
    random_state=42,
    n_jobs=-1
)

rf_random_search.fit(X_train, y_train)

print('Parameter Terbaik RF:', rf_random_search.best_params_)
print('RMSE Terbaik RF:', np.sqrt(-rf_random_search.best_score_))

best_rf_model = rf_random_search.best_estimator_

In [ ]:
best_rf_model.fit(X_train, y_train)

rf_log_predictions = best_rf_model.predict(X_test)
rf_final_predictions = np.expm1(rf_log_predictions)

rf_submission = pd.DataFrame({'Id': test_ID, 'SalePrice': rf_final_predictions})
rf_submission.to_csv('../outputs/submission_rf.csv', index=False)

joblib.dump(best_rf_model, '../models/model_rf_rumah.pkl')
print('Random Forest model saved as models/model_rf_rumah.pkl')
print('Submission saved as submission_rf.csv')

### EVALUATION ###

In [ ]:
from sklearn.model_selection import cross_validate
from sklearn.metrics import mean_absolute_error, r2_score, mean_absolute_percentage_error, mean_squared_error

target_asli = np.expm1(y_train)
std_dev_saleprice = target_asli.std()
print(f'Rata-rata Harga Rumah  : {target_asli.mean():,.2f}')
print(f'Standard Deviation     : {std_dev_saleprice:,.2f}')

def rmse_scorer(model, X, y_log):
    return np.sqrt(mean_squared_error(np.expm1(y_log), np.expm1(model.predict(X))))

def mae_scorer(model, X, y_log):
    return mean_absolute_error(np.expm1(y_log), np.expm1(model.predict(X)))

def r2_scorer(model, X, y_log):
    return r2_score(np.expm1(y_log), np.expm1(model.predict(X)))

def mape_scorer(model, X, y_log):
    return mean_absolute_percentage_error(np.expm1(y_log), np.expm1(model.predict(X)))

scoring_custom = {'rmse': rmse_scorer, 'mae': mae_scorer, 'r2': r2_scorer, 'mape': mape_scorer}

#### XGBoost Evaluation

In [ ]:
cv_xgb = cross_validate(best_xgb_model, X_train, y_train, cv=kf, scoring=scoring_custom)

mean_rmse = cv_xgb['test_rmse'].mean()
mean_mae  = cv_xgb['test_mae'].mean()
mean_r2   = cv_xgb['test_r2'].mean()
mean_mape = cv_xgb['test_mape'].mean()

print('=' * 50)
print('EVALUASI XGBOOST:')
print(f'RMSE : {mean_rmse:,.2f}')
print(f'MAE  : {mean_mae:,.2f}')
print(f'R2   : {mean_r2:.4f}')
print(f'MAPE : {mean_mape:.4f}')
print('=' * 50)

from sklearn.metrics import r2_score
r2_train_xgb = r2_score(np.expm1(y_train), np.expm1(best_xgb_model.predict(X_train)))
print(f'R2 Training : {r2_train_xgb:.4f} | R2 Validasi : {mean_r2:.4f}')
if r2_train_xgb - mean_r2 > 0.10:
    print('TERINDIKASI OVERFITTING!')
else:
    print('AMAN: Tidak overfit.')

#### Random Forest Evaluation

In [ ]:
cv_rf = cross_validate(best_rf_model, X_train, y_train, cv=kf, scoring=scoring_custom)

rf_mean_rmse = cv_rf['test_rmse'].mean()
rf_mean_mae  = cv_rf['test_mae'].mean()
rf_mean_r2   = cv_rf['test_r2'].mean()
rf_mean_mape = cv_rf['test_mape'].mean()

print('=' * 50)
print('EVALUASI RANDOM FOREST:')
print(f'RMSE : {rf_mean_rmse:,.2f}')
print(f'MAE  : {rf_mean_mae:,.2f}')
print(f'R2   : {rf_mean_r2:.4f}')
print(f'MAPE : {rf_mean_mape:.4f}')
print('=' * 50)

r2_train_rf = r2_score(np.expm1(y_train), np.expm1(best_rf_model.predict(X_train)))
print(f'R2 Training : {r2_train_rf:.4f} | R2 Validasi : {rf_mean_r2:.4f}')
if r2_train_rf - rf_mean_r2 > 0.10:
    print('TERINDIKASI OVERFITTING!')
else:
    print('AMAN: Tidak overfit.')

### MODEL COMPARISON & MVP MODEL ###

In [ ]:
print('=' * 50)
print('PERBANDINGAN MODEL:')
print(f'XGBoost      → RMSE: {mean_rmse:,.2f} | R2: {mean_r2:.4f}')
print(f'Random Forest → RMSE: {rf_mean_rmse:,.2f} | R2: {rf_mean_r2:.4f}')
print('=' * 50)

if mean_rmse <= rf_mean_rmse:
    best_model = best_xgb_model
    winner = 'XGBoost'
else:
    best_model = best_rf_model
    winner = 'Random Forest'

joblib.dump(best_model, '../final_model.pkl')
print(f'{winner} menang! Disimpan sebagai final_model.pkl')

final_predictions = np.expm1(best_model.predict(X_test))
submission = pd.DataFrame({'Id': test_ID, 'SalePrice': final_predictions})
submission.to_csv('../outputs/submission_terbaik.csv', index=False)
print('Final submission saved as outputs/submission_terbaik.csv')

print('\nModel files saved:')
print('  models/model_xgboost_rumah.pkl  → XGBoost tuned')
print('  models/model_rf_rumah.pkl       → Random Forest tuned')
print('  final_model.pkl                → Best model (', winner, ')')